# Predicción de semanas con alta demanda respiratoria usando NOx y urgencias MINSAL

Flujo reproducible de Machine Learning en Databricks/MLflow para clasificar semanas con alta demanda de atenciones respiratorias en la Región de Valparaíso.

Fuentes:
- SINCA: registros diarios de NOx por estación.
- MINSAL: atenciones de urgencia, causa `IdCausa = 2`, total de causas del sistema respiratorio.

La unidad de análisis es semanal (`anio`, `semana`). El objetivo es predictivo y exploratorio; no permite afirmar causalidad sanitaria.


## 1. Imports y configuración


In [0]:
from functools import reduce
import gc
import os
import re
import unicodedata

import mlflow
from mlflow.models.signature import infer_signature

from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array

In [0]:
CATALOG = "upla"
SCHEMA = "mcdma_analisis_datos_ma"
VOLUME = "aire_salud_volume"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SINCA_DIR = f"{BASE_PATH}/raw/sinca"
URGENCIAS_DIR = f"{BASE_PATH}/raw/urgencias"
MLFLOW_TMP = f"{BASE_PATH}/tmp/mlflow"

BRONZE_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sinca_nox_diario"
BRONZE_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_urgencias"
SILVER_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.silver_sinca_nox_semanal"
SILVER_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.silver_urgencias_resp_semanal"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_aire_salud_semanal"
FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.features_aire_salud_semanal"
MODEL_COMPARISON_TABLE = f"{CATALOG}.{SCHEMA}.model_comparison_aire_salud_nox"
PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.predictions_aire_salud_nox"

EXPERIMENT_NAME = f"/Shared/{CATALOG}_{SCHEMA}_aire_salud_nox_modelos"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.modelo_alta_demanda_respiratoria_nox"
REGISTER_MODEL = False

TRAIN_END_YEAR = 2024
TEST_YEAR = 2025
HIGH_DEMAND_QUANTILE = 0.75
RANDOM_SEED = 42

os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)

print("BASE_PATH:", BASE_PATH)
print("SINCA_DIR:", SINCA_DIR)
print("URGENCIAS_DIR:", URGENCIAS_DIR)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)


BASE_PATH: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume
SINCA_DIR: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca
URGENCIAS_DIR: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias
EXPERIMENT_NAME: /Shared/upla_mcdma_analisis_datos_ma_aire_salud_nox_modelos


In [0]:
display(dbutils.fs.ls(SINCA_DIR))
display(dbutils.fs.ls(URGENCIAS_DIR))


path,name,size,modificationTime
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_centro_quintero_2018_2025.csv,v_nox_diario_centro_quintero_2018_2025.csv,66851,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_colmo_2018_2025.csv,v_nox_diario_colmo_2018_2025.csv,65893,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_concon_2018_2025.csv,v_nox_diario_concon_2018_2025.csv,66632,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_cuerpo_de_bomberos_2018_2025.csv,v_nox_diario_cuerpo_de_bomberos_2018_2025.csv,65744,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_greda_2018_2025.csv,v_nox_diario_la_greda_2018_2025.csv,66696,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_palma_2018_2025.csv,v_nox_diario_la_palma_2018_2025.csv,65295,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_loncura_2018_2025.csv,v_nox_diario_loncura_2018_2025.csv,63205,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_los_andes_2018_2025.csv,v_nox_diario_los_andes_2018_2025.csv,62665,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_los_maitenes_2018_2025.csv,v_nox_diario_los_maitenes_2018_2025.csv,66687,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_puchuncavi_2018_2025.csv,v_nox_diario_puchuncavi_2018_2025.csv,66693,1781149870000


path,name,size,modificationTime
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2020.csv,AtencionesUrgencia2020.csv,796227186,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2021.csv,AtencionesUrgencia2021.csv,1130831814,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2022.csv,AtencionesUrgencia2022.csv,1144770884,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2023.csv,AtencionesUrgencia2023.csv,1598833998,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2024.csv,AtencionesUrgencia2024.csv,1608911448,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2025.csv,AtencionesUrgencia2025.csv,1784414515,1781149926000


## 2. Funciones auxiliares


In [0]:
def clean_col_name(c):
    c = c.strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
    c = re.sub(r"[^0-9A-Za-z]+", "_", c)
    c = c.strip("_").lower()
    return c


def make_unique_columns(cols):
    clean_cols = []
    seen = {}
    for i, c in enumerate(cols):
        name = clean_col_name(c) or f"empty_col_{i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        clean_cols.append(name)
    return clean_cols


def select_first_existing(df, candidates, alias_name):
    for c in candidates:
        if c in df.columns:
            return F.col(c).alias(alias_name)
    return F.lit(None).alias(alias_name)


def to_double_comma(col_name):
    return (
        F.regexp_replace(
            F.when(F.trim(F.col(col_name).cast("string")) == "", None)
             .otherwise(F.trim(F.col(col_name).cast("string"))),
            ",",
            ".",
        )
        .cast("double")
    )


def to_long_safe(col_name):
    return to_double_comma(col_name).cast("long")


def read_csv_folder(folder_path, encoding="UTF-8"):
    files = [f.path for f in dbutils.fs.ls(folder_path) if f.name.lower().endswith(".csv")]
    print("Archivos encontrados:", len(files))
    dfs = []
    for path in files:
        print("Leyendo:", path.split("/")[-1])
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("sep", ";")
            .option("encoding", encoding)
            .option("inferSchema", "false")
            .load(path)
        )
        df = df.toDF(*make_unique_columns(df.columns))
        dfs.append(df.withColumn("source_file", F.lit(path.split("/")[-1])))

    all_cols = sorted(set().union(*[set(df.columns) for df in dfs]))
    dfs_aligned = [
        df.select([F.col(c) if c in df.columns else F.lit(None).alias(c) for c in all_cols])
        for df in dfs
    ]
    return reduce(lambda a, b: a.unionByName(b), dfs_aligned)


def read_urgencias_file(path, encoding="ISO-8859-1"):
    print("Leyendo:", path.split("/")[-1])
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", encoding)
        .option("inferSchema", "false")
        .load(path)
    )
    df = df.toDF(*make_unique_columns(df.columns))
    df = df.select(
        select_first_existing(df, ["idestablecimiento", "id_establecimiento"], "idestablecimiento"),
        select_first_existing(df, ["nestablecimiento", "n_establecimiento", "nombre_establecimiento"], "nestablecimiento"),
        select_first_existing(df, ["idcausa", "id_causa"], "idcausa"),
        select_first_existing(df, ["glosacausa", "glosa_causa"], "glosacausa"),
        select_first_existing(df, ["total"], "total"),
        select_first_existing(df, ["menores_1", "menor_a_1", "menor_1"], "menores_1"),
        select_first_existing(df, ["de_1_a_4", "column7"], "de_1_a_4"),
        select_first_existing(df, ["de_5_a_14", "_14"], "de_5_a_14"),
        select_first_existing(df, ["de_15_a_64", "_5_64"], "de_15_a_64"),
        select_first_existing(df, ["de_65_y_mas", "_5_mas", "de_65_y_mas_"], "de_65_y_mas"),
        select_first_existing(df, ["fecha"], "fecha"),
        select_first_existing(df, ["semana"], "semana"),
        select_first_existing(df, ["glosatipoestablecimiento"], "glosatipoestablecimiento"),
        select_first_existing(df, ["glosatipoatencion"], "glosatipoatencion"),
        select_first_existing(df, ["glosatipocampana"], "glosatipocampana"),
        select_first_existing(df, ["codigoregion", "codigo_region"], "codigoregion"),
        select_first_existing(df, ["nombreregion", "nombre_region"], "nombreregion"),
        select_first_existing(df, ["codigodependencia", "codigo_dependencia"], "codigodependencia"),
        select_first_existing(df, ["nombredependencia", "nombre_dependencia"], "nombredependencia"),
        select_first_existing(df, ["codigocomuna", "codigo_comuna"], "codigocomuna"),
        select_first_existing(df, ["nombrecomuna", "nombre_comuna"], "nombrecomuna"),
    )
    return (
        df.withColumn("source_file", F.lit(path.split("/")[-1]))
        .withColumn("fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy"))
        .withColumn("anio", F.year("fecha"))
        .withColumn("semana_minsal", F.col("semana").cast("int"))
        .withColumn("idcausa_int", F.col("idcausa").cast("int"))
        .withColumn("codigoregion_int", F.col("codigoregion").cast("int"))
        .withColumn("codigocomuna_int", F.col("codigocomuna").cast("int"))
        .withColumn("total_num", to_long_safe("total"))
        .withColumn("menores_1_num", to_long_safe("menores_1"))
        .withColumn("de_1_a_4_num", to_long_safe("de_1_a_4"))
        .withColumn("de_5_a_14_num", to_long_safe("de_5_a_14"))
        .withColumn("de_15_a_64_num", to_long_safe("de_15_a_64"))
        .withColumn("de_65_y_mas_num", to_long_safe("de_65_y_mas"))
        .filter(F.col("fecha").isNotNull())
        .filter((F.col("anio") >= 2020) & (F.col("anio") <= 2025))
    )


## 3. Carga y limpieza SINCA


In [0]:
sinca_raw = read_csv_folder(SINCA_DIR, encoding="UTF-8")
print("Columnas SINCA:", sinca_raw.columns)
display(sinca_raw.limit(10))


Archivos encontrados: 15
Leyendo: v_nox_diario_centro_quintero_2018_2025.csv
Leyendo: v_nox_diario_colmo_2018_2025.csv
Leyendo: v_nox_diario_concon_2018_2025.csv
Leyendo: v_nox_diario_cuerpo_de_bomberos_2018_2025.csv
Leyendo: v_nox_diario_la_greda_2018_2025.csv
Leyendo: v_nox_diario_la_palma_2018_2025.csv
Leyendo: v_nox_diario_loncura_2018_2025.csv
Leyendo: v_nox_diario_los_andes_2018_2025.csv
Leyendo: v_nox_diario_los_maitenes_2018_2025.csv
Leyendo: v_nox_diario_puchuncavi_2018_2025.csv
Leyendo: v_nox_diario_quintero_2018_2025.csv
Leyendo: v_nox_diario_san_pedro_2018_2025.csv
Leyendo: v_nox_diario_sur_2018_2025.csv
Leyendo: v_nox_diario_valle_alegre_2018_2025.csv
Leyendo: v_nox_diario_ventanas_2018_2025.csv
Columnas SINCA: ['c5', 'fecha_yymmdd', 'hora_hhmm', 'registros_no_validados', 'registros_preliminares', 'registros_validados', 'source_file']


c5,fecha_yymmdd,hora_hhmm,registros_no_validados,registros_preliminares,registros_validados,source_file
null,180101,0000,"9,03348",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180102,0000,"12,2869",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180103,0000,"14,4776",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180104,0000,"16,9356",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180105,0000,"4,82915",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180106,0000,"11,1457",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180107,0000,"7,0108",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180108,0000,"10,5179",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180109,0000,"10,0118",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180110,0000,"7,57923",null,null,v_nox_diario_centro_quintero_2018_2025.csv


In [0]:
sinca_clean = (
    sinca_raw
    .withColumn("fecha_str", F.lpad(F.col("fecha_yymmdd").cast("string"), 6, "0"))
    .withColumn(
        "fecha",
        F.to_date(
            F.concat(
                F.lit("20"),
                F.substring("fecha_str", 1, 2),
                F.substring("fecha_str", 3, 2),
                F.substring("fecha_str", 5, 2),
            ),
            "yyyyMMdd",
        ),
    )
    .withColumn(
        "estacion",
        F.regexp_replace(
            F.regexp_replace(F.col("source_file"), r"^v_nox_diario_", ""),
            r"_2018_2025\.csv$",
            "",
        ),
    )
    .withColumn("valor_validado", to_double_comma("registros_validados"))
    .withColumn("valor_preliminar", to_double_comma("registros_preliminares"))
    .withColumn("valor_no_validado", to_double_comma("registros_no_validados"))
    .withColumn("valor_nox", F.coalesce("valor_validado", "valor_preliminar", "valor_no_validado"))
    .withColumn(
        "tipo_registro",
        F.when(F.col("valor_validado").isNotNull(), F.lit("validado"))
         .when(F.col("valor_preliminar").isNotNull(), F.lit("preliminar"))
         .when(F.col("valor_no_validado").isNotNull(), F.lit("no_validado"))
         .otherwise(F.lit("sin_dato")),
    )
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("valor_nox").isNotNull())
    .filter((F.col("fecha") >= F.lit("2020-01-01")) & (F.col("fecha") <= F.lit("2025-12-31")))
    .select("fecha", "estacion", F.lit("NOx").alias("contaminante"), "valor_nox", "tipo_registro", "source_file")
)

(
    sinca_clean.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_SINCA_TABLE)
)

print("Tabla creada:", BRONZE_SINCA_TABLE)
display(sinca_clean.limit(20))


Tabla creada: upla.mcdma_analisis_datos_ma.bronze_sinca_nox_diario


fecha,estacion,contaminante,valor_nox,tipo_registro,source_file
2020-01-01,centro_quintero,NOx,6.19751,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-02,centro_quintero,NOx,8.90251,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-03,centro_quintero,NOx,6.38358,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-04,centro_quintero,NOx,6.74619,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-05,centro_quintero,NOx,6.12192,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-06,centro_quintero,NOx,6.59077,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-07,centro_quintero,NOx,9.35983,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-08,centro_quintero,NOx,6.2711,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-09,centro_quintero,NOx,12.3439,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-10,centro_quintero,NOx,10.0713,no_validado,v_nox_diario_centro_quintero_2018_2025.csv


In [0]:
display(sinca_clean.groupBy("tipo_registro").count().orderBy(F.desc("count")))

display(
    sinca_clean
    .groupBy("estacion")
    .agg(
        F.count("*").alias("n_registros"),
        F.min("fecha").alias("fecha_min"),
        F.max("fecha").alias("fecha_max"),
        F.avg("valor_nox").alias("nox_promedio"),
        F.max("valor_nox").alias("nox_maximo"),
    )
    .orderBy("estacion")
)


tipo_registro,count
no_validado,29669
validado,1850
preliminar,661


estacion,n_registros,fecha_min,fecha_max,nox_promedio,nox_maximo
centro_quintero,2184,2020-01-01,2025-12-30,14.377350123626387,54.7701
colmo,2162,2020-01-01,2025-12-30,14.63443065217391,84.2017
concon,2153,2020-01-01,2025-12-30,17.51840104505341,96.2332
cuerpo_de_bomberos,2074,2020-01-01,2025-12-30,19.92206397299907,71.4554
la_greda,2161,2020-01-01,2025-12-30,15.87636989356782,61.5365
la_palma,2070,2020-01-01,2025-12-30,9.44697606280194,61.3524
loncura,2139,2020-01-01,2025-12-30,8.841431425899941,40.925
los_andes,2167,2020-01-01,2025-12-30,14.111477798800212,67.9076
los_maitenes,2153,2020-01-01,2025-12-30,9.791911560613109,34.5511
puchuncavi,2159,2020-01-01,2025-12-30,16.473149620194548,49.4931


## 4. Carga, normalización geográfica y filtro MINSAL


In [0]:
urg_files = [f.path for f in dbutils.fs.ls(URGENCIAS_DIR) if f.name.lower().endswith(".csv")]
print("Archivos de urgencias:", len(urg_files))
for f in urg_files:
    print(f)

urg_dfs = [read_urgencias_file(path, encoding="ISO-8859-1") for path in urg_files]
urg_all = reduce(lambda a, b: a.unionByName(b), urg_dfs)

print("DataFrame urg_all creado")
display(urg_all.limit(10))


Archivos de urgencias: 6
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2020.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2021.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2022.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2023.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2024.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2025.csv
Leyendo: AtencionesUrgencia2020.csv
Leyendo: AtencionesUrgencia2021.csv
Leyendo: AtencionesUrgencia2022.csv
Leyendo: AtencionesUrgencia2023.csv
Leyendo: AtencionesUrgencia2024.csv
Leyendo: AtencionesUrgencia2025.csv
DataFrame urg_all creado


idestablecimiento,nestablecimiento,idcausa,glosacausa,total,menores_1,de_1_a_4,de_5_a_14,de_15_a_64,de_65_y_mas,fecha,semana,glosatipoestablecimiento,glosatipoatencion,glosatipocampana,codigoregion,nombreregion,codigodependencia,nombredependencia,codigocomuna,nombrecomuna,source_file,anio,semana_minsal,idcausa_int,codigoregion_int,codigocomuna_int,total_num,menores_1_num,de_1_a_4_num,de_5_a_14_num,de_15_a_64_num,de_65_y_mas_num
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,22,0,1,4,15,2,2020-09-23,39,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,39,21,null,null,22,0,1,4,15,2
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,30,0,2,3,21,4,2020-09-24,39,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,39,21,null,null,30,0,2,3,21,4
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,9,0,2,1,4,2,2020-09-25,39,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,39,21,null,null,9,0,2,1,4,2
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,17,0,0,1,14,2,2020-09-26,39,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,39,21,null,null,17,0,0,1,14,2
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,22,2,0,2,14,4,2020-09-27,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,22,2,0,2,14,4
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,22,0,0,4,15,3,2020-09-28,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,22,0,0,4,15,3
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,11,0,1,0,9,1,2020-09-29,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,11,0,1,0,9,1
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,15,0,2,0,12,1,2020-09-30,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,15,0,2,0,12,1
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,20,0,0,4,13,3,2020-10-01,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,20,0,0,4,13,3
19-809,SAPU Talcahuano Sur,21,TOTAL DEMÁS CAUSAS,17,0,1,2,14,0,2020-10-02,40,SAPU,Indiferenciado,Ninguna,null,null,null,null,null,null,AtencionesUrgencia2020.csv,2020,40,21,null,null,17,0,1,2,14,0


In [0]:
display(
    urg_all
    .groupBy("anio", "source_file")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_int").isNotNull(), 1).otherwise(0)).alias("filas_con_region"),
        F.sum(F.when(F.col("codigocomuna_int").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna"),
    )
    .orderBy("anio", "source_file")
)


anio,source_file,filas,filas_con_region,filas_con_comuna
2020,AtencionesUrgencia2020.csv,6446646,0,0
2021,AtencionesUrgencia2021.csv,8816240,0,0
2022,AtencionesUrgencia2022.csv,8926307,0,0
2023,AtencionesUrgencia2023.csv,8899080,8899080,8899080
2024,AtencionesUrgencia2024.csv,8973229,8965509,8965509
2025,AtencionesUrgencia2025.csv,9142479,9142239,9142239


In [0]:
establecimientos_base = (
    urg_all
    .filter(F.col("codigoregion_int").isNotNull())
    .filter(F.col("codigocomuna_int").isNotNull())
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
    .groupBy("idestablecimiento_norm", "codigoregion_int", "nombreregion", "codigocomuna_int", "nombrecomuna")
    .agg(F.count("*").alias("n_apariciones"))
)

w_est = Window.partitionBy("idestablecimiento_norm").orderBy(F.desc("n_apariciones"))

establecimientos_map = (
    establecimientos_base
    .withColumn("rn", F.row_number().over(w_est))
    .filter(F.col("rn") == 1)
    .select(
        "idestablecimiento_norm",
        F.col("codigoregion_int").alias("map_codigoregion"),
        F.col("nombreregion").alias("map_nombreregion"),
        F.col("codigocomuna_int").alias("map_codigocomuna"),
        F.col("nombrecomuna").alias("map_nombrecomuna"),
    )
)

urg_all_filled = (
    urg_all
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
    .alias("u")
    .join(establecimientos_map.alias("m"), on="idestablecimiento_norm", how="left")
    .withColumn("codigoregion_final", F.coalesce(F.col("u.codigoregion_int"), F.col("m.map_codigoregion")))
    .withColumn("nombreregion_final", F.coalesce(F.col("u.nombreregion"), F.col("m.map_nombreregion")))
    .withColumn("codigocomuna_final", F.coalesce(F.col("u.codigocomuna_int"), F.col("m.map_codigocomuna")))
    .withColumn("nombrecomuna_final", F.coalesce(F.col("u.nombrecomuna"), F.col("m.map_nombrecomuna")))
)

urg_valpo = (
    urg_all_filled
    .filter((F.col("codigoregion_final") == 5) | (F.upper(F.col("nombreregion_final")).contains("VALPAR")))
)

(
    urg_valpo.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_URGENCIAS_TABLE)
)

print("Establecimientos mapeados:", establecimientos_map.count())
print("Tabla creada:", BRONZE_URGENCIAS_TABLE)


Establecimientos mapeados: 667
Tabla creada: upla.mcdma_analisis_datos_ma.bronze_urgencias


In [0]:
display(
    urg_all_filled
    .groupBy("anio")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_final").isNotNull(), 1).otherwise(0)).alias("filas_con_region_final"),
        F.sum(F.when(F.col("codigocomuna_final").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna_final"),
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos"),
    )
    .orderBy("anio")
)

display(
    urg_valpo
    .groupBy("anio")
    .agg(
        F.count("*").alias("filas_valpo"),
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos_valpo"),
        F.countDistinct("codigocomuna_final").alias("n_comunas_valpo"),
        F.sum("total_num").alias("total_atenciones"),
    )
    .orderBy("anio")
)


anio,filas,filas_con_region_final,filas_con_comuna_final,n_establecimientos
2020,6446646,6338147,6338147,607
2021,8816240,8747080,8747080,635
2022,8926307,8919840,8919840,638
2023,8899080,8899080,8899080,642
2024,8973229,8973229,8973229,640
2025,9142479,9142439,9142439,653


anio,filas_valpo,n_establecimientos_valpo,n_comunas_valpo,total_atenciones
2020,587189,52,25,3130660
2021,818560,67,30,4297648
2022,897320,73,34,6848463
2023,911680,74,34,6822155
2024,904760,71,34,6870795
2025,957673,76,36,6759210


In [0]:
resp_candidates = (
    urg_valpo
    .filter(
        F.upper(F.col("glosacausa")).contains("RESPIRATOR")
        | F.upper(F.col("glosacausa")).contains("BRONQUIT")
        | F.upper(F.col("glosacausa")).contains("NEUMON")
        | F.upper(F.col("glosacausa")).contains("INFLUENZA")
        | F.upper(F.col("glosacausa")).contains("IRA")
    )
    .groupBy("anio", "idcausa_int", "glosacausa")
    .agg(F.sum("total_num").alias("total"), F.count("*").alias("filas"))
    .orderBy("anio", "idcausa_int")
)

display(resp_candidates)


anio,idcausa_int,glosacausa,total,filas
2020,2,TOTAL CAUSAS SISTEMA RESPIRATORIO,109990,16582
2020,3,Bronquitis/bronquiolitis aguda (J20-J21),9898,16582
2020,4,Influenza (J09-J11),754,16582
2020,5,Neumonía (J12-J18),4762,16582
2020,6,"Otra causa respiratoria (J22, J30-J39, J47, J60-J98)",13280,16582
2020,7,CAUSAS SISTEMA RESPIRATORIO,2970,16582
2020,10,IRA Alta (J00-J06),73477,16582
2021,2,TOTAL CAUSAS SISTEMA RESPIRATORIO,130365,20464
2021,3,Bronquitis/bronquiolitis aguda (J20-J21),10648,20464
2021,4,Influenza (J09-J11),462,20464


## 5. Agregación semanal SINCA y MINSAL


In [0]:
urg_resp = (
    urg_valpo
    .filter(F.col("idcausa_int") == 2)
    .filter(F.col("semana_minsal").between(1, 53))
    .filter(F.col("total_num").isNotNull())
)

silver_urgencias = (
    urg_resp
    .groupBy(F.col("anio"), F.col("semana_minsal").alias("semana"))
    .agg(
        F.min("fecha").alias("fecha_inicio_semana"),
        F.max("fecha").alias("fecha_fin_semana"),
        F.sum("total_num").alias("total_resp_urgencia"),
        F.sum("menores_1_num").alias("resp_menores_1"),
        F.sum("de_1_a_4_num").alias("resp_1_a_4"),
        F.sum("de_5_a_14_num").alias("resp_5_a_14"),
        F.sum("de_15_a_64_num").alias("resp_15_a_64"),
        F.sum("de_65_y_mas_num").alias("resp_65_y_mas"),
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos_urg"),
        F.countDistinct("codigocomuna_final").alias("n_comunas_urg"),
        F.count("*").alias("n_filas_urg"),
    )
)

(
    silver_urgencias.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_URGENCIAS_TABLE)
)

print("Tabla creada:", SILVER_URGENCIAS_TABLE)
display(silver_urgencias.orderBy("anio", "semana"))


Tabla creada: upla.mcdma_analisis_datos_ma.silver_urgencias_resp_semanal


anio,semana,fecha_inicio_semana,fecha_fin_semana,total_resp_urgencia,resp_menores_1,resp_1_a_4,resp_5_a_14,resp_15_a_64,resp_65_y_mas,n_establecimientos_urg,n_comunas_urg,n_filas_urg
2020,1,2020-01-01,2020-01-04,2549,198,442,363,1239,307,44,23,163
2020,2,2020-01-05,2020-01-11,4139,315,636,578,2029,581,42,22,279
2020,3,2020-01-12,2020-01-18,3658,278,636,526,1804,414,41,22,279
2020,4,2020-01-19,2020-01-25,3714,258,597,510,1858,491,40,21,280
2020,5,2020-01-26,2020-02-01,3474,232,605,545,1669,423,40,21,272
2020,6,2020-02-02,2020-02-08,3726,269,535,565,1953,404,41,22,280
2020,7,2020-02-09,2020-02-15,3968,291,574,610,2029,464,41,22,283
2020,8,2020-02-16,2020-02-22,4139,267,613,654,2137,468,41,22,286
2020,9,2020-02-23,2020-02-29,4300,277,613,630,2289,491,42,22,283
2020,10,2020-03-01,2020-03-07,4355,241,567,620,2431,496,44,23,294


In [0]:
silver_sinca = (
    sinca_clean
    .withColumn("anio", F.year("fecha"))
    .withColumn("semana", F.weekofyear("fecha"))
    .filter(F.col("semana").between(1, 53))
    .groupBy("anio", "semana")
    .agg(
        F.min("fecha").alias("fecha_inicio_nox"),
        F.max("fecha").alias("fecha_fin_nox"),
        F.avg("valor_nox").alias("nox_prom_sem"),
        F.max("valor_nox").alias("nox_max_sem"),
        F.expr("percentile_approx(valor_nox, 0.95)").alias("nox_p95_sem"),
        F.stddev("valor_nox").alias("nox_std_sem"),
        F.count("*").alias("n_obs_nox"),
        F.countDistinct("estacion").alias("n_estaciones_nox"),
        F.sum(F.when(F.col("tipo_registro") == "validado", 1).otherwise(0)).alias("n_obs_validadas"),
        F.sum(F.when(F.col("tipo_registro") == "preliminar", 1).otherwise(0)).alias("n_obs_preliminares"),
        F.sum(F.when(F.col("tipo_registro") == "no_validado", 1).otherwise(0)).alias("n_obs_no_validadas"),
    )
)

(
    silver_sinca.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_SINCA_TABLE)
)

print("Tabla creada:", SILVER_SINCA_TABLE)
display(silver_sinca.orderBy("anio", "semana"))


Tabla creada: upla.mcdma_analisis_datos_ma.silver_sinca_nox_semanal


anio,semana,fecha_inicio_nox,fecha_fin_nox,nox_prom_sem,nox_max_sem,nox_p95_sem,nox_std_sem,n_obs_nox,n_estaciones_nox,n_obs_validadas,n_obs_preliminares,n_obs_no_validadas
2020,1,2020-01-01,2020-01-05,7.283832266666667,13.0138,12.477,2.317250443056337,75,15,5,15,55
2020,2,2020-01-06,2020-01-12,9.058983076923077,21.7047,15.4195,3.289512548436399,104,15,7,21,76
2020,3,2020-01-13,2020-01-19,8.731462596153845,20.9076,16.0837,3.6742819454281155,104,15,7,21,76
2020,4,2020-01-20,2020-01-26,7.405787403846154,16.5476,12.8147,2.8647837492662833,104,15,7,21,76
2020,5,2020-01-27,2020-02-02,8.17104980769231,18.7943,13.2115,3.1357330924678553,104,15,7,20,77
2020,6,2020-02-03,2020-02-09,10.534148181818182,23.5166,18.9381,4.352794458435695,99,15,7,16,76
2020,7,2020-02-10,2020-02-16,9.329437572815534,22.8775,15.7889,3.936474045612559,103,15,7,19,77
2020,8,2020-02-17,2020-02-23,9.656308952380954,22.6642,15.5651,3.7920693615998737,105,15,7,21,77
2020,9,2020-02-24,2020-03-01,9.926507142857144,23.3103,18.5575,4.192139519966377,98,15,5,16,77
2020,10,2020-03-02,2020-03-08,11.033535904761905,25.0236,17.3559,3.976061308612286,105,15,7,21,77


## 6. Unión Gold, variables derivadas y creación del label


In [0]:
gold_base = (
    silver_urgencias.alias("u")
    .join(silver_sinca.alias("s"), on=["anio", "semana"], how="left")
    .withColumn("mes", F.month("fecha_inicio_semana"))
    .withColumn(
        "estacion_anio",
        F.when(F.col("mes").isin(12, 1, 2), F.lit("verano"))
         .when(F.col("mes").isin(3, 4, 5), F.lit("otono"))
         .when(F.col("mes").isin(6, 7, 8), F.lit("invierno"))
         .when(F.col("mes").isin(9, 10, 11), F.lit("primavera"))
         .otherwise(F.lit("desconocida")),
    )
    .withColumn("periodo_covid", F.when(F.col("anio").isin(2020, 2021), F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn("serie_region", F.lit("valparaiso"))
)

w_week = Window.partitionBy("serie_region").orderBy("anio", "semana")

gold_with_lags = (
    gold_base
    .withColumn("nox_prom_lag1", F.lag("nox_prom_sem", 1).over(w_week))
    .withColumn("nox_max_lag1", F.lag("nox_max_sem", 1).over(w_week))
    .withColumn("nox_p95_lag1", F.lag("nox_p95_sem", 1).over(w_week))
    .withColumn("nox_prom_lag2", F.lag("nox_prom_sem", 2).over(w_week))
    .drop("serie_region")
)

threshold_row = (
    gold_with_lags
    .filter(F.col("anio") <= TRAIN_END_YEAR)
    .select(F.expr(f"percentile_approx(total_resp_urgencia, {HIGH_DEMAND_QUANTILE})").alias("threshold"))
    .first()
)

HIGH_DEMAND_THRESHOLD = float(threshold_row["threshold"])
print("Umbral alta demanda respiratoria P75 train:", HIGH_DEMAND_THRESHOLD)

gold = (
    gold_with_lags
    .withColumn("alta_demanda_threshold", F.lit(HIGH_DEMAND_THRESHOLD))
    .withColumn(
        "label",
        F.when(F.col("total_resp_urgencia") > F.lit(HIGH_DEMAND_THRESHOLD), F.lit(1.0)).otherwise(F.lit(0.0)),
    )
)

(
    gold.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

gold = spark.table(GOLD_TABLE)

print("Tabla creada:", GOLD_TABLE)
display(gold.orderBy("anio", "semana"))


Umbral alta demanda respiratoria P75 train: 10356.0
Tabla creada: upla.mcdma_analisis_datos_ma.gold_aire_salud_semanal


anio,semana,fecha_inicio_semana,fecha_fin_semana,total_resp_urgencia,resp_menores_1,resp_1_a_4,resp_5_a_14,resp_15_a_64,resp_65_y_mas,n_establecimientos_urg,n_comunas_urg,n_filas_urg,fecha_inicio_nox,fecha_fin_nox,nox_prom_sem,nox_max_sem,nox_p95_sem,nox_std_sem,n_obs_nox,n_estaciones_nox,n_obs_validadas,n_obs_preliminares,n_obs_no_validadas,mes,estacion_anio,periodo_covid,nox_prom_lag1,nox_max_lag1,nox_p95_lag1,nox_prom_lag2,alta_demanda_threshold,label
2020,1,2020-01-01,2020-01-04,2549,198,442,363,1239,307,44,23,163,2020-01-01,2020-01-05,7.283832266666667,13.0138,12.477,2.317250443056337,75,15,5,15,55,1,verano,1.0,null,null,null,null,10356.0,0.0
2020,2,2020-01-05,2020-01-11,4139,315,636,578,2029,581,42,22,279,2020-01-06,2020-01-12,9.058983076923077,21.7047,15.4195,3.289512548436399,104,15,7,21,76,1,verano,1.0,7.283832266666667,13.0138,12.477,null,10356.0,0.0
2020,3,2020-01-12,2020-01-18,3658,278,636,526,1804,414,41,22,279,2020-01-13,2020-01-19,8.731462596153845,20.9076,16.0837,3.6742819454281155,104,15,7,21,76,1,verano,1.0,9.058983076923077,21.7047,15.4195,7.283832266666667,10356.0,0.0
2020,4,2020-01-19,2020-01-25,3714,258,597,510,1858,491,40,21,280,2020-01-20,2020-01-26,7.405787403846154,16.5476,12.8147,2.8647837492662833,104,15,7,21,76,1,verano,1.0,8.731462596153845,20.9076,16.0837,9.058983076923077,10356.0,0.0
2020,5,2020-01-26,2020-02-01,3474,232,605,545,1669,423,40,21,272,2020-01-27,2020-02-02,8.17104980769231,18.7943,13.2115,3.1357330924678553,104,15,7,20,77,1,verano,1.0,7.405787403846154,16.5476,12.8147,8.731462596153845,10356.0,0.0
2020,6,2020-02-02,2020-02-08,3726,269,535,565,1953,404,41,22,280,2020-02-03,2020-02-09,10.534148181818182,23.5166,18.9381,4.352794458435695,99,15,7,16,76,2,verano,1.0,8.17104980769231,18.7943,13.2115,7.405787403846154,10356.0,0.0
2020,7,2020-02-09,2020-02-15,3968,291,574,610,2029,464,41,22,283,2020-02-10,2020-02-16,9.329437572815534,22.8775,15.7889,3.936474045612559,103,15,7,19,77,2,verano,1.0,10.534148181818182,23.5166,18.9381,8.17104980769231,10356.0,0.0
2020,8,2020-02-16,2020-02-22,4139,267,613,654,2137,468,41,22,286,2020-02-17,2020-02-23,9.656308952380954,22.6642,15.5651,3.7920693615998737,105,15,7,21,77,2,verano,1.0,9.329437572815534,22.8775,15.7889,10.534148181818182,10356.0,0.0
2020,9,2020-02-23,2020-02-29,4300,277,613,630,2289,491,42,22,283,2020-02-24,2020-03-01,9.926507142857144,23.3103,18.5575,4.192139519966377,98,15,5,16,77,2,verano,1.0,9.656308952380954,22.6642,15.5651,9.329437572815534,10356.0,0.0
2020,10,2020-03-01,2020-03-07,4355,241,567,620,2431,496,44,23,294,2020-03-02,2020-03-08,11.033535904761905,25.0236,17.3559,3.976061308612286,105,15,7,21,77,3,otono,1.0,9.926507142857144,23.3103,18.5575,9.656308952380954,10356.0,0.0


In [0]:
display(
    gold
    .groupBy("anio", "label")
    .agg(
        F.count("*").alias("semanas"),
        F.avg("total_resp_urgencia").alias("promedio_urgencias"),
        F.avg("nox_prom_sem").alias("promedio_nox"),
    )
    .orderBy("anio", "label")
)


anio,label,semanas,promedio_urgencias,promedio_nox
2020,0.0,53,2075.2830188679245,11.783258963606386
2021,0.0,53,2459.7169811320755,13.183288974721064
2022,0.0,35,7462.0,13.668963682466238
2022,1.0,17,12675.70588235294,15.123519659887382
2023,0.0,28,7461.25,12.652207326250252
2023,1.0,24,13157.0,14.136135689541929
2024,0.0,28,7232.321428571428,11.71054446269542
2024,1.0,24,12691.166666666666,14.007668308442002
2025,0.0,33,7194.69696969697,12.550115826210927
2025,1.0,20,11061.6,14.532063282330558


## 7. Preparación de features y división temporal


In [0]:
num_cols = [
    "semana",
    "mes",
    "periodo_covid",
    "nox_prom_sem",
    "nox_max_sem",
    "nox_p95_sem",
    "nox_std_sem",
    "n_obs_nox",
    "n_estaciones_nox",
    "nox_prom_lag1",
    "nox_max_lag1",
    "nox_p95_lag1",
    "nox_prom_lag2",
]

cat_cols = ["estacion_anio"]

id_cols = [
    "anio",
    "semana",
    "fecha_inicio_semana",
    "fecha_fin_semana",
    "total_resp_urgencia",
    "alta_demanda_threshold",
]

feature_cols = list(dict.fromkeys(id_cols + ["label"] + num_cols + cat_cols))

gold_for_features = spark.table(GOLD_TABLE)

df_features = (
    gold_for_features
    .select(*[c for c in feature_cols if c in gold_for_features.columns])
    .withColumn("estacion_anio", F.coalesce(F.col("estacion_anio"), F.lit("desconocida")))
    .filter(F.col("label").isNotNull())
)

(
    df_features.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FEATURES_TABLE)
)

train_df = df_features.filter(F.col("anio") <= TRAIN_END_YEAR)
test_df = df_features.filter(F.col("anio") == TEST_YEAR)

print("Filas train:", train_df.count())
print("Filas test:", test_df.count())
display(train_df.groupBy("label").count().orderBy("label"))
display(test_df.groupBy("label").count().orderBy("label"))


Filas train: 262
Filas test: 53


label,count
0.0,197
1.0,65


label,count
0.0,33
1.0,20


## 8. Modelos candidatos


In [0]:
def build_pipeline(estimator):
    imputed_cols = [f"{c}_imp" for c in num_cols]

    imputer = Imputer(inputCols=num_cols, outputCols=imputed_cols, strategy="median")
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    encoder = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in cat_cols],
        outputCols=[f"{c}_ohe" for c in cat_cols],
        handleInvalid="keep",
    )
    assembler = VectorAssembler(
        inputCols=imputed_cols + [f"{c}_ohe" for c in cat_cols],
        outputCol="features",
        handleInvalid="keep",
    )
    return Pipeline(stages=[imputer] + indexers + [encoder, assembler, estimator])


# En Serverless/Spark Connect conviene mantener pocos candidatos y registrar
# como artefacto solo el mejor modelo. Esto evita exceder la cache ML de la sesion.
models_to_test = {
    "logistic_regression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=50,
        regParam=0.01,
    ),
    "decision_tree": DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=4,
        maxBins=32,
        seed=RANDOM_SEED,
    ),
}

print("Modelos a evaluar:", list(models_to_test.keys()))

Modelos a evaluar: ['logistic_regression', 'decision_tree']


## 9. Evaluación y funciones MLflow


In [0]:
def compute_auc(preds, label_col="label"):
    evaluator = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    return evaluator.evaluate(preds)


def compute_multiclass_metric(preds, metric_name, label_col="label", pred_col="prediction"):
    evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=pred_col,
        metricName=metric_name,
    )
    return evaluator.evaluate(preds)


def compute_confusion_matrix(preds, label_col="label", pred_col="prediction"):
    return (
        preds
        .select(F.col(label_col).cast("int").alias("label"), F.col(pred_col).cast("int").alias("prediction"))
        .groupBy("label", "prediction")
        .count()
        .orderBy("label", "prediction")
    )


def add_probability_positive(preds, probability_col="probability", output_col="prob_alta_demanda"):
    return preds.withColumn(output_col, vector_to_array(F.col(probability_col))[1].cast("double"))


def cleanup_ml_objects():
    variable_names = [
        "fitted_model",
        "best_model",
        "pipeline",
        "best_pipeline",
        "train_preds",
        "test_preds",
        "full_preds",
        "best_test_preds",
        "output_example",
        "signature",
    ]
    for variable_name in variable_names:
        if variable_name in globals():
            del globals()[variable_name]
    gc.collect()
    # En Databricks Serverless, CLEAR CACHE no esta soportado.
    # Este notebook no usa cache()/persist(), por lo que basta liberar referencias Python.


input_example = train_df.select(num_cols + cat_cols).limit(5).toPandas()
cleanup_ml_objects()


## 10. Entrenamiento, comparación y registro en MLflow

Para evitar el límite de caché de modelos en Databricks Serverless/Spark Connect, los modelos candidatos registran métricas y matriz de confusión, pero no se guardan todos como artefactos Spark ML. Después de comparar resultados, se vuelve a entrenar solo el mejor modelo y ese único modelo se registra con firma de entrada/salida.

In [0]:
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_registry_uri("databricks-uc")


In [0]:
comparison_rows = []
best_result = None
best_model_name = None
best_run_id = None
best_model_artifact_run_id = None
best_model_uri = None
registered_model_version = None
full_preds = None
best_test_preds = None

with mlflow.start_run(run_name="aire_salud_nox_model_comparison") as parent_run:
    parent_run_id = parent_run.info.run_id

    mlflow.log_param("dataset", "SINCA NOx + MINSAL urgencias respiratorias")
    mlflow.log_param("region", "Valparaiso")
    mlflow.log_param("unit_of_analysis", "anio_semana")
    mlflow.log_param("label_definition", f"total_resp_urgencia > P{int(HIGH_DEMAND_QUANTILE * 100)} train")
    mlflow.log_param("high_demand_threshold", HIGH_DEMAND_THRESHOLD)
    mlflow.log_param("train_period", f"2020-{TRAIN_END_YEAR}")
    mlflow.log_param("test_period", str(TEST_YEAR))
    mlflow.log_param("categorical_features", ",".join(cat_cols))
    mlflow.log_param("numerical_features", ",".join(num_cols))
    mlflow.log_param("model_candidates", ",".join(models_to_test.keys()))
    mlflow.log_param("selection_metric", "test_auc_then_f1")
    mlflow.log_param("registered_model_name", MODEL_NAME if REGISTER_MODEL else "not_registered")
    mlflow.log_param("serverless_strategy", "log_metrics_for_candidates_log_only_best_model")

    for candidate_name, estimator in models_to_test.items():
        cleanup_ml_objects()
        with mlflow.start_run(run_name=f"{candidate_name}_metrics", nested=True) as child_run:
            child_run_id = child_run.info.run_id
            pipeline = build_pipeline(estimator)
            fitted_model = pipeline.fit(train_df)

            train_preds = add_probability_positive(fitted_model.transform(train_df))
            test_preds = add_probability_positive(fitted_model.transform(test_df))

            train_auc = compute_auc(train_preds)
            test_auc = compute_auc(test_preds)
            test_accuracy = compute_multiclass_metric(test_preds, "accuracy")
            test_f1 = compute_multiclass_metric(test_preds, "f1")
            test_precision = compute_multiclass_metric(test_preds, "weightedPrecision")
            test_recall = compute_multiclass_metric(test_preds, "weightedRecall")
            test_cm_df = compute_confusion_matrix(test_preds)

            mlflow.log_param("model_type", candidate_name)
            mlflow.log_param("logged_spark_model", False)
            mlflow.log_metric("train_auc", train_auc)
            mlflow.log_metric("test_auc", test_auc)
            mlflow.log_metric("test_accuracy", test_accuracy)
            mlflow.log_metric("test_f1", test_f1)
            mlflow.log_metric("test_precision_weighted", test_precision)
            mlflow.log_metric("test_recall_weighted", test_recall)
            mlflow.log_table(test_cm_df.toPandas(), "confusion_matrix_test.json")

            current_result = {
                "model_name": candidate_name,
                "run_id": child_run_id,
                "model_uri": "logged_only_if_selected",
                "train_auc": float(train_auc),
                "test_auc": float(test_auc),
                "test_accuracy": float(test_accuracy),
                "test_f1": float(test_f1),
                "test_precision_weighted": float(test_precision),
                "test_recall_weighted": float(test_recall),
            }
            comparison_rows.append(current_result)

            if best_result is None or (current_result["test_auc"], current_result["test_f1"]) > (best_result["test_auc"], best_result["test_f1"]):
                best_result = current_result
                best_model_name = candidate_name
                best_run_id = child_run_id

            print(f"Modelo evaluado: {candidate_name} | AUC test: {test_auc:.4f} | Accuracy: {test_accuracy:.4f} | F1: {test_f1:.4f}")

        cleanup_ml_objects()

    mlflow.log_param("best_model_name", best_model_name)
    mlflow.log_param("best_model_metrics_run_id", best_run_id)
    mlflow.log_metric("best_test_auc", best_result["test_auc"])
    mlflow.log_metric("best_test_accuracy", best_result["test_accuracy"])
    mlflow.log_metric("best_test_f1", best_result["test_f1"])

    cleanup_ml_objects()
    with mlflow.start_run(run_name=f"{best_model_name}_best_model_artifact", nested=True) as model_run:
        best_model_artifact_run_id = model_run.info.run_id
        best_pipeline = build_pipeline(models_to_test[best_model_name])
        best_model = best_pipeline.fit(train_df)

        full_preds = add_probability_positive(best_model.transform(df_features))
        best_test_preds = add_probability_positive(best_model.transform(test_df))

        output_example = (
            best_model
            .transform(spark.createDataFrame(input_example))
            .withColumn("prob_alta_demanda", vector_to_array(F.col("probability"))[1].cast("double"))
            .select(
                F.col("prediction").cast("double").alias("prediction"),
                F.col("prob_alta_demanda").cast("double").alias("prob_alta_demanda"),
            )
            .toPandas()
        )
        signature = infer_signature(input_example, output_example)

        mlflow.log_param("model_type", best_model_name)
        mlflow.log_param("source_metrics_run_id", best_run_id)
        mlflow.log_param("logged_spark_model", True)
        mlflow.log_metric("test_auc", best_result["test_auc"])
        mlflow.log_metric("test_accuracy", best_result["test_accuracy"])
        mlflow.log_metric("test_f1", best_result["test_f1"])

        mlflow.spark.log_model(
            spark_model=best_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            dfs_tmpdir=MLFLOW_TMP,
        )

        best_model_uri = f"runs:/{best_model_artifact_run_id}/model"

        if REGISTER_MODEL:
            registered_model = mlflow.register_model(model_uri=best_model_uri, name=MODEL_NAME)
            registered_model_version = registered_model.version
            mlflow.log_param("registered_model_version", registered_model_version)

    mlflow.log_param("best_model_artifact_run_id", best_model_artifact_run_id)
    mlflow.log_param("best_model_uri", best_model_uri)

comparison_df = spark.createDataFrame(comparison_rows)
(
    comparison_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MODEL_COMPARISON_TABLE)
)

print("Parent Run ID:", parent_run_id)
print("Mejor modelo:", best_model_name)
print("Best metrics Run ID:", best_run_id)
print("Best artifact Run ID:", best_model_artifact_run_id)
print("Best model URI:", best_model_uri)
print("Best Test AUC:", best_result["test_auc"])
print("Best Test Accuracy:", best_result["test_accuracy"])
print("Best Test F1:", best_result["test_f1"])
print("Modelo registrado en:", MODEL_NAME if REGISTER_MODEL else "Solo guardado como artefacto MLflow")
print("Version registrada:", registered_model_version)

display(comparison_df.orderBy(F.desc("test_auc"), F.desc("test_f1")))
display(compute_confusion_matrix(best_test_preds))
display(full_preds.select("anio", "semana", "label", "prediction", "prob_alta_demanda").orderBy("anio", "semana"))

Modelo evaluado: logistic_regression | AUC test: 0.8258 | Accuracy: 0.7736 | F1: 0.7736
Modelo evaluado: decision_tree | AUC test: 0.8106 | Accuracy: 0.7547 | F1: 0.7579


/opt/databricks-environments/databricks-ml/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/11 08:31:05 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.6) contains a local version label (+databricks.connect.18.0.6). MLflow logged a pip requirement for th

Parent Run ID: 681da95ec8af410e84bc91e0bcc2775a
Mejor modelo: logistic_regression
Best metrics Run ID: a06d8fb129c0422783cd2e92245df4eb
Best artifact Run ID: f0a7c08e98f44a42aa4b1eadce1bd048
Best model URI: runs:/f0a7c08e98f44a42aa4b1eadce1bd048/model
Best Test AUC: 0.8257575757575757
Best Test Accuracy: 0.7735849056603774
Best Test F1: 0.7735849056603774
Modelo registrado en: Solo guardado como artefacto MLflow
Version registrada: None


model_name,model_uri,run_id,test_accuracy,test_auc,test_f1,test_precision_weighted,test_recall_weighted,train_auc
logistic_regression,logged_only_if_selected,a06d8fb129c0422783cd2e92245df4eb,0.7735849056603774,0.8257575757575757,0.7735849056603774,0.7735849056603774,0.7735849056603774,0.9311987504880905
decision_tree,logged_only_if_selected,062521e8abaf4442b705fb78c6d6f3fc,0.7547169811320755,0.8106060606060606,0.7578785238027683,0.8049772283669486,0.7547169811320755,0.9409996095275284


label,prediction,count
0,0,27
0,1,6
1,0,6
1,1,14


anio,semana,label,prediction,prob_alta_demanda
2020,1,0.0,0.0,3.9881878186598385E-5
2020,2,0.0,0.0,5.128184841878269E-4
2020,3,0.0,0.0,3.464787350442311E-4
2020,4,0.0,0.0,3.8670777366034415E-4
2020,5,0.0,0.0,3.851561673394732E-4
2020,6,0.0,0.0,2.873514572999669E-4
2020,7,0.0,0.0,3.9402495934204307E-4
2020,8,0.0,0.0,5.358898545039237E-4
2020,9,0.0,0.0,3.1192618043729414E-4
2020,10,0.0,0.0,0.016897997847061363


## 11. Guardado de predicciones


In [0]:
cols_to_drop = ["features", "rawPrediction", "probability"]
cols_to_drop += [f"{c}_idx" for c in cat_cols]
cols_to_drop += [f"{c}_ohe" for c in cat_cols]
cols_to_drop += [f"{c}_imp" for c in num_cols]

preds_with_metadata = (
    full_preds
    .drop(*[c for c in cols_to_drop if c in full_preds.columns])
    .withColumn("model_run_id", F.lit(best_model_artifact_run_id))
    .withColumn("selected_model_type", F.lit(best_model_name))
    .withColumn("registered_model_name", F.lit(MODEL_NAME if REGISTER_MODEL else ""))
    .withColumn("registered_model_version", F.lit(str(registered_model_version) if registered_model_version else ""))
    .withColumn("scoring_timestamp", F.current_timestamp())
)

ordered_cols = list(dict.fromkeys([
    "anio",
    "semana",
    "fecha_inicio_semana",
    "fecha_fin_semana",
    "total_resp_urgencia",
    "alta_demanda_threshold",
    "label",
    "prediction",
    "prob_alta_demanda",
] + num_cols + cat_cols + [
    "model_run_id",
    "selected_model_type",
    "registered_model_name",
    "registered_model_version",
    "scoring_timestamp",
]))

preds_with_metadata = preds_with_metadata.select(*[c for c in ordered_cols if c in preds_with_metadata.columns])

(
    preds_with_metadata.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PREDICTIONS_TABLE)
)

print("Tabla creada:", PREDICTIONS_TABLE)
display(preds_with_metadata.orderBy("anio", "semana"))

Tabla creada: upla.mcdma_analisis_datos_ma.predictions_aire_salud_nox


anio,semana,fecha_inicio_semana,fecha_fin_semana,total_resp_urgencia,alta_demanda_threshold,label,prediction,prob_alta_demanda,mes,periodo_covid,nox_prom_sem,nox_max_sem,nox_p95_sem,nox_std_sem,n_obs_nox,n_estaciones_nox,nox_prom_lag1,nox_max_lag1,nox_p95_lag1,nox_prom_lag2,estacion_anio,model_run_id,selected_model_type,registered_model_name,registered_model_version,scoring_timestamp
2020,1,2020-01-01,2020-01-04,2549,10356.0,0.0,0.0,3.9881878186598385E-5,1,1.0,7.283832266666667,13.0138,12.477,2.317250443056337,75,15,null,null,null,null,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,2,2020-01-05,2020-01-11,4139,10356.0,0.0,0.0,5.128184841878269E-4,1,1.0,9.058983076923077,21.7047,15.4195,3.289512548436399,104,15,7.283832266666667,13.0138,12.477,null,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,3,2020-01-12,2020-01-18,3658,10356.0,0.0,0.0,3.464787350442311E-4,1,1.0,8.731462596153845,20.9076,16.0837,3.6742819454281155,104,15,9.058983076923077,21.7047,15.4195,7.283832266666667,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,4,2020-01-19,2020-01-25,3714,10356.0,0.0,0.0,3.8670777366034415E-4,1,1.0,7.405787403846154,16.5476,12.8147,2.8647837492662833,104,15,8.731462596153845,20.9076,16.0837,9.058983076923077,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,5,2020-01-26,2020-02-01,3474,10356.0,0.0,0.0,3.851561673394732E-4,1,1.0,8.17104980769231,18.7943,13.2115,3.1357330924678553,104,15,7.405787403846154,16.5476,12.8147,8.731462596153845,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,6,2020-02-02,2020-02-08,3726,10356.0,0.0,0.0,2.873514572999669E-4,2,1.0,10.534148181818182,23.5166,18.9381,4.352794458435695,99,15,8.17104980769231,18.7943,13.2115,7.405787403846154,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,7,2020-02-09,2020-02-15,3968,10356.0,0.0,0.0,3.9402495934204307E-4,2,1.0,9.329437572815534,22.8775,15.7889,3.936474045612559,103,15,10.534148181818182,23.5166,18.9381,8.17104980769231,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,8,2020-02-16,2020-02-22,4139,10356.0,0.0,0.0,5.358898545039237E-4,2,1.0,9.656308952380954,22.6642,15.5651,3.7920693615998737,105,15,9.329437572815534,22.8775,15.7889,10.534148181818182,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,9,2020-02-23,2020-02-29,4300,10356.0,0.0,0.0,3.1192618043729414E-4,2,1.0,9.926507142857144,23.3103,18.5575,4.192139519966377,98,15,9.656308952380954,22.6642,15.5651,9.329437572815534,verano,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z
2020,10,2020-03-01,2020-03-07,4355,10356.0,0.0,0.0,0.016897997847061363,3,1.0,11.033535904761905,25.0236,17.3559,3.976061308612286,105,15,9.926507142857144,23.3103,18.5575,9.656308952380954,otono,f0a7c08e98f44a42aa4b1eadce1bd048,logistic_regression,,,2026-06-11T08:31:58.363Z


## 12. Validación final y evidencia para informe


In [0]:
print("Tablas generadas:")
for table_name in [
    BRONZE_SINCA_TABLE,
    BRONZE_URGENCIAS_TABLE,
    SILVER_SINCA_TABLE,
    SILVER_URGENCIAS_TABLE,
    GOLD_TABLE,
    FEATURES_TABLE,
    MODEL_COMPARISON_TABLE,
    PREDICTIONS_TABLE,
]:
    print(table_name, spark.table(table_name).count())


Tablas generadas:
upla.mcdma_analisis_datos_ma.bronze_sinca_nox_diario 32180
upla.mcdma_analisis_datos_ma.bronze_urgencias 5077182
upla.mcdma_analisis_datos_ma.silver_sinca_nox_semanal 314
upla.mcdma_analisis_datos_ma.silver_urgencias_resp_semanal 315
upla.mcdma_analisis_datos_ma.gold_aire_salud_semanal 315
upla.mcdma_analisis_datos_ma.features_aire_salud_semanal 315
upla.mcdma_analisis_datos_ma.model_comparison_aire_salud_nox 2
upla.mcdma_analisis_datos_ma.predictions_aire_salud_nox 315


## 13. Limitaciones metodológicas

- El análisis es predictivo y exploratorio; no permite inferir causalidad entre NOx y atenciones respiratorias.
- Una proporción relevante de registros SINCA puede estar clasificada como no validada; esto debe reportarse explícitamente.
- La agregación semanal reduce ruido, pero también oculta variabilidad diaria y posibles rezagos más finos.
- El cruce semanal usa la semana calculada desde la fecha en SINCA y la semana reportada por MINSAL; en una versión posterior conviene usar un calendario epidemiológico común.
- No se incorporan todavía meteorología, circulación atmosférica, virus respiratorios, movilidad ni teledetección satelital.
- Para una tesis, este flujo puede ampliarse con ERA5, Sentinel-5P/TROPOMI, rezagos distribuidos y análisis espacial por comuna o área de influencia de estaciones.
